<a href="https://colab.research.google.com/github/ritvik-123/EMP-Project/blob/master/Module_1_SmallData_Rework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentence-Level Oppression-Type Classification — Small-Data Rework

This notebook replaces the single train/test split + 2-stage cascade from
`Module_1_26.ipynb` with an approach better suited to **154 sentences / 26
groups**:

1. Smaller sentence embeddings (`bge-small-en-v1.5`, 384-dim) instead of
   `bge-large` (1024-dim) — fewer dimensions relative to your sample size.
2. **GroupKFold cross-validation** (grouped by `source_id`) instead of one
   holdout split, so every sentence gets evaluated out-of-fold and you get a
   distribution of scores instead of a single lucky/unlucky number.
3. **One single-stage multi-label classifier** instead of a gate + type
   cascade — removes the "train on an already-small subset of an
   already-small dataset" problem.
4. A **nearest-centroid baseline** trained the same way, so you can see
   whether logistic regression is actually earning its extra complexity.

Run top to bottom. Every line has a comment explaining what it does and why.


In [123]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [124]:
# --- Imports -----------------------------------------------------------
import pandas as pd                                   # tabular data handling
import numpy as np                                     # array math

from sentence_transformers import SentenceTransformer  # turns sentences into embedding vectors

from sklearn.model_selection import GroupKFold         # CV splitter that keeps a group's rows together
from sklearn.decomposition import TruncatedSVD         # dimensionality reduction (SVD works with dense/sparse)
from sklearn.linear_model import LogisticRegression    # our main classifier
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier     # wraps LogisticRegression to handle multi-label output
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, top_k_accuracy_score, f1_score     # precision/recall/F1 per label
from sklearn.preprocessing import normalize             # L2-normalizes vectors (needed for cosine similarity)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import torch

from datasets import Dataset
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)



In [125]:
# --- Config --------------------------------------------------------------
CSV_PATH = "../Data/"

MODEL_PATH = "../Model/"   # your labeled sentence-level file

TARGET_LABELS = [                                # the 4 label columns we're predicting
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized",
]
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"  # 384-dim model: smaller than bge-large, less overfitting risk
N_SVD_COMPONENTS = 200                            # compress 384-dim embeddings down further before the classifier
N_FOLDS = 5                                      # number of GroupKFold splits (you could also use LeaveOneGroupOut)
RANDOM_STATE = 42                                # fixed seed so results are reproducible
THRESHOLD = 0.5                                  # probability cutoff for turning a score into a 0/1 prediction


In [126]:
%pwd
%cd /content/drive/MyDrive/EMP/Notebook

/content/drive/MyDrive/EMP/Notebook


## 1. Load and clean the data

In [127]:
# --- Load -----------------------------------------------------------------
df = pd.read_csv(CSV_PATH + "Generated_Sentences_1.csv")

df = df[df['data_source'] != 'extra_institutionalized']

# Clean column names just in case there are spaces
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())

# Clean sentences
df["Sentence"] = df["Sentence"].fillna("").astype(str).str.strip()
df = df[df["Sentence"] != ""].reset_index(drop=True)

# Clean Label column
df["Label"] = df["Label"].fillna("").astype(str).str.strip().str.lower()

# Create the 4 target label columns from the single Label column
for label in TARGET_LABELS:
    df[label] = (df["Label"] == label).astype(int)

# Optional: create source_id if your new file does not have one
if "source_id" not in df.columns:
    df["source_id"] = df.index

print("Rows after cleaning:", len(df))
print("Unique groups (source_id):", df["source_id"].nunique())
print(df[TARGET_LABELS].sum())

Columns: ['Sentence', 'Label', 'data_source']
Rows after cleaning: 550
Unique groups (source_id): 550
ideological          150
institutionalized    150
interpersonal        150
internalized         100
dtype: int64


In [128]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 2. Embed the sentences

We embed the raw sentence text once, up front, for the whole dataset. Note
that fitting the classifier still happens *inside* each CV fold — the
embedding step itself doesn't "see" the labels, so computing it once outside
the loop is not a leak.

In [129]:
# --- Embed ------------------------------------------------------------------
device = "cuda"   # change to "cpu" if you're not on a GPU runtime

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)   # loads the pretrained embedding model

sentences = df["Sentence"].tolist()                                   # plain list of strings to embed

X_full = embedder.encode(
    sentences,                     # the sentences to embed
    batch_size=32,                 # how many sentences to embed per forward pass
    normalize_embeddings=True,     # L2-normalize each embedding (helps cosine-similarity-style comparisons)
    show_progress_bar=True,        # display a progress bar since this can take a minute
    convert_to_numpy=True,         # return a numpy array instead of torch tensors
)

print("Embedding matrix shape:", X_full.shape)   # expect (num_sentences, 384)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Embedding matrix shape: (550, 1024)


## 3. Reduce dimensionality with SVD

384 dimensions on ~150 training rows is still a lot of parameters relative to
data. `TruncatedSVD` compresses the embeddings to `N_SVD_COMPONENTS` dims,
which reduces overfitting risk in the downstream logistic regression.

**Important**: SVD is fit *inside* each CV fold (see the loop below), not on
the whole dataset up front — fitting it globally first would leak
information from the test fold into training.

## 4. Prepare labels and groups for cross-validation

In [130]:
# --- Targets and groups -----------------------------------------------------
Y = df[TARGET_LABELS].values          # shape (n_sentences, 4) — the multi-label target matrix
groups = df["source_id"].values       # which participant/source each sentence belongs to

# GroupKFold ensures all sentences from the same source_id land in the same
# fold — so the model is never trained and tested on sentences from the same
# person, which would make scores look better than they really are.
gkf = GroupKFold(n_splits=N_FOLDS)


In [131]:
df.head()

,Sentence,Label,data_source,ideological,institutionalized,interpersonal,internalized,source_id
0,I was raised around the idea that speaking Eng...,ideological,base_synthetic,1,0,0,0,0
1,People often treat whiteness as the default st...,ideological,base_synthetic,1,0,0,0,1
2,She learned early that darker skin was describ...,ideological,base_synthetic,1,0,0,0,2
3,The assumption that men are naturally better l...,ideological,base_synthetic,1,0,0,0,3
4,I noticed how poverty was framed as a personal...,ideological,base_synthetic,1,0,0,0,4


## 5. Cross-validated logistic regression (single-stage, multi-label)

Instead of a gate stage + a separate type stage, this trains **one**
`OneVsRestClassifier(LogisticRegression)` per fold on all 4 labels at once. A
sentence with all-zero predictions is implicitly "no strong leaning" — you
don't need a separate gate model for that.

In [132]:
# --- Cross-validated training loop -------------------------------------------
oof_proba_lr = np.zeros_like(Y, dtype=float)   # will hold out-of-fold predicted probabilities for every row

fold_reports = []   # collect per-fold classification reports so we can see variance across folds

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_full, Y, groups=groups)):
    # --- split this fold's data ---
    X_train_raw, X_test_raw = X_full[train_idx], X_full[test_idx]   # raw 384-dim embeddings for train/test rows
    Y_train, Y_test = Y[train_idx], Y[test_idx]                     # matching label rows

    # --- fit SVD on TRAIN ONLY, then apply to both train and test ---
    svd = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=RANDOM_STATE)  # dimensionality reducer
    X_train = svd.fit_transform(X_train_raw)   # learn the SVD components from training embeddings only
    X_test = svd.transform(X_test_raw)         # apply the same learned components to the test embeddings

    # X_train = X_train_raw  # learn the SVD components from training embeddings only
    # X_test = X_test_raw

    # --- fit the classifier on this fold's training data ---
    clf = OneVsRestClassifier(
        LogisticRegression(
            max_iter=2000,              # more iterations so the solver reliably converges
            class_weight="balanced",    # up-weights rare positive labels automatically
            solver="liblinear",         # solver that works well for small/medium binary problems
            random_state=RANDOM_STATE,  # reproducibility
        )
    )
    clf.fit(X_train, Y_train)                      # train on this fold's training rows

    # --- predict probabilities on the held-out fold ---
    test_proba = clf.predict_proba(X_test)          # shape (n_test_rows, 4) — probability per label
    oof_proba_lr[test_idx] = test_proba             # store these as this fold's out-of-fold predictions

    # --- per-fold report, just to see how much folds vary ---
    fold_preds = (test_proba >= THRESHOLD).astype(int)
    fold_reports.append(
        classification_report(Y_test, fold_preds, target_names=TARGET_LABELS,
                               zero_division=0, output_dict=True)
    )
    print(f"Fold {fold_idx}: {len(test_idx)} test rows, {len(train_idx)} train rows")

print("\nDone. Every row now has an out-of-fold probability it never trained on.")


Fold 0: 110 test rows, 440 train rows
Fold 1: 110 test rows, 440 train rows
Fold 2: 110 test rows, 440 train rows
Fold 3: 110 test rows, 440 train rows
Fold 4: 110 test rows, 440 train rows

Done. Every row now has an out-of-fold probability it never trained on.


In [133]:
# ------------------------------------------------------------
# Final model trained on ALL generated sentences
# ------------------------------------------------------------


y_train_full = df["Label"].values

final_svd = TruncatedSVD(
    n_components=N_SVD_COMPONENTS,
    random_state=RANDOM_STATE
)

sample_weight = np.ones(len(df))

sample_weight[df["data_source"] == "base_synthetic"] = 1.0
sample_weight[df["data_source"] == "contrastive_synthetic"] = 0.7
sample_weight[df["data_source"] == "extra_institutionalized"] = 0.4

X_train_full = final_svd.fit_transform(X_full)

final_clf = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    class_weight=None,
    random_state=RANDOM_STATE
)

final_clf.fit(X_train_full, y_train_full, sample_weight=sample_weight)

print("Final model trained on all generated sentences.")
print("Classes:", final_clf.classes_)

Final model trained on all generated sentences.
Classes: ['ideological' 'institutionalized' 'internalized' 'interpersonal']


## 6. Aggregate results across all folds

This is the single number/report you should actually cite — every sentence
was scored by a model that never saw it during training, and it pools all
26 groups instead of relying on one lucky/unlucky split.

In [134]:
df.head()

,Sentence,Label,data_source,ideological,institutionalized,interpersonal,internalized,source_id
0,I was raised around the idea that speaking Eng...,ideological,base_synthetic,1,0,0,0,0
1,People often treat whiteness as the default st...,ideological,base_synthetic,1,0,0,0,1
2,She learned early that darker skin was describ...,ideological,base_synthetic,1,0,0,0,2
3,The assumption that men are naturally better l...,ideological,base_synthetic,1,0,0,0,3
4,I noticed how poverty was framed as a personal...,ideological,base_synthetic,1,0,0,0,4


In [135]:
# --- Pooled out-of-fold evaluation -------------------------------------------
oof_preds_lr = (oof_proba_lr >= THRESHOLD).astype(int)   # threshold the pooled probabilities into 0/1

print("=== Logistic Regression: pooled out-of-fold report (all 26 groups) ===")
print(classification_report(Y, oof_preds_lr, target_names=TARGET_LABELS, zero_division=0))


=== Logistic Regression: pooled out-of-fold report (all 26 groups) ===
                   precision    recall  f1-score   support

      ideological       0.74      0.80      0.77       150
institutionalized       0.91      0.95      0.93       150
    interpersonal       0.81      0.94      0.87       150
     internalized       0.72      0.97      0.83       100

        micro avg       0.80      0.91      0.85       550
        macro avg       0.80      0.92      0.85       550
     weighted avg       0.80      0.91      0.85       550
      samples avg       0.83      0.91      0.85       550



In [136]:
# --- Per-fold variance check --------------------------------------------------
# Pull out each fold's macro-F1 so you can see how much a single split would
# have swung your reported number.
macro_f1_per_fold = [r["macro avg"]["f1-score"] for r in fold_reports]   # macro-F1 for each of the 5 folds

print("Macro-F1 per fold:", np.round(macro_f1_per_fold, 3))
print("Mean macro-F1:", np.round(np.mean(macro_f1_per_fold), 3))
print("Std  macro-F1:", np.round(np.std(macro_f1_per_fold), 3))
print("--> report mean ± std, not a single split's number, in any writeup.")


Macro-F1 per fold: [0.905 0.804 0.856 0.856 0.832]
Mean macro-F1: 0.851
Std  macro-F1: 0.033
--> report mean ± std, not a single split's number, in any writeup.


## 7. Training on Linear SVC

In [137]:
# ------------------------------------------------------------
# Final Linear SVC model trained on ALL generated sentences
# ------------------------------------------------------------


y_train_full = df["Label"].values

final_svd_svc = TruncatedSVD(
    n_components=N_SVD_COMPONENTS,
    random_state=RANDOM_STATE
)

X_train_full_svc = final_svd_svc.fit_transform(X_full)

sample_weight = np.ones(len(df))

sample_weight[df["data_source"] == "base_synthetic"] = 1.0
sample_weight[df["data_source"] == "contrastive_synthetic"] = 0.7
sample_weight[df["data_source"] == "extra_institutionalized"] = 0.4

final_svc = LinearSVC(
    C=1.0,
    class_weight=None,
    random_state=RANDOM_STATE
)

final_svc.fit(X_train_full_svc, y_train_full, sample_weight=sample_weight)

print("Final Linear SVC model trained on all generated sentences.")
print("Classes:", final_svc.classes_)

Final Linear SVC model trained on all generated sentences.
Classes: ['ideological' 'institutionalized' 'internalized' 'interpersonal']


## Notes on what changed vs. the original notebook

- **No single held-out split** — every sentence contributes to both training
  (in 4 of 5 folds) and evaluation (in its 1 held-out fold), and the report
  you cite is pooled across all folds, not one split.
- **No gate + type cascade** — one multi-label model handles "is this
  oppression-related" and "which type" simultaneously; an all-zero row is
  implicitly "no strong leaning."
- **SVD is refit inside each fold** — this avoids leaking test-fold
  information into the dimensionality reduction step.
- **A trivial baseline (nearest centroid) is reported alongside the real
  model** — at n≈150 this is the honest way to show your logistic
  regression is earning its complexity, not just fitting noise.


## Eval on Real sentences

Evaluation on sentences that are real responses from real people and to test the model's credibility on real world data.

In [138]:
df_eval = pd.read_csv(CSV_PATH + "Module 1 Sentences Gemini.csv")
df_eval

,source_id,campus,sentence_index,sentence_id,sentence,ideological,ideological_score,institutionalized,institutionalized_score,interpersonal,interpersonal_score,internalized,internalized_score,primary_leaning,rationale,gemini_raw,reviewed,review_notes
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...,0,0.0,0,0.0,0,0.0,1,0.7,internalized,The sentence reflects on a shift in self-perce...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
1,1,Chico,1,1_1,I think it is important to know how each indiv...,0,0.0,0,0.0,1,0.7,0,0.0,interpersonal,The sentence emphasizes the importance of indi...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence describes differing life experien...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
3,1,Chico,3,1_3,Since we all have so many different identities...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence discusses the complexity of inter...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
4,1,Chico,4,1_4,I want to continue listening to my students in...,0,0.0,1,0.7,0,0.0,0,0.0,institutionalized,The sentence discusses interpreting generalize...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,22,San Francisco,0,22_0,"Age: 56 years, my son calls me old a number of...",0,0.0,0,0.0,1,0.7,0,0.0,interpersonal,The sentence describes a person being called '...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
150,23,San Francisco,0,23_0,"Status: Lecturer Faculty, Instructor",0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence simply states a job title and doe...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
151,24,San Francisco,0,24_0,Gender: Male,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence is a simple statement of gender a...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
152,25,San Francisco,0,25_0,Disability: Wearing eye glasses,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence simply states a fact about disabi...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN


In [139]:
# dropping all the unwanted rows and columns
df_eval = df_eval[['sentence','ideological','institutionalized','interpersonal','internalized','primary_leaning']]
df_eval = df_eval[df_eval['primary_leaning'] != 'none']
df_eval = df_eval.reset_index(drop=True)
df_eval = df_eval.rename(columns={'sentence':'Sentence'})
df_eval

,Sentence,ideological,institutionalized,interpersonal,internalized,primary_leaning
0,Something that I noticed is how I feel about m...,0,0,0,1,internalized
1,I think it is important to know how each indiv...,0,0,1,0,interpersonal
2,I want to continue listening to my students in...,0,1,0,0,institutionalized
3,An insight I gained from this activity has to ...,1,0,0,0,ideological
4,"However, this activity forced me to consider t...",1,0,0,0,ideological
5,"Coming here, I was color blind.",1,0,0,0,ideological
6,I soon learned to see color.,1,0,0,0,ideological
7,I learned to see the struggles of minority gro...,1,0,1,1,internalized
8,I experienced how it felt to be discriminated ...,1,0,1,0,ideological
9,This exercise has brought back some difficult ...,0,0,0,1,internalized


In [140]:
# make sure primary_leaning has no weird spaces/case issues
df_eval['primary_leaning'] = (
    df_eval['primary_leaning']
    .astype(str)
    .str.strip()
    .str.lower()
)

# true single-label values
y_true = df_eval['primary_leaning'].values

# embed sentences
X_eval_raw = embedder.encode(
    df_eval["Sentence"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [141]:
# IMPORTANT: use final_svd, not the last fold's svd
X_eval = final_svd.transform(X_eval_raw)
#X_eval = X_eval_raw

# IMPORTANT: use final_clf, not the last fold's clf
eval_proba_raw = final_clf.predict_proba(X_eval)

# Align probability columns to TARGET_LABELS order
class_to_col = {label: i for i, label in enumerate(final_clf.classes_)}

eval_proba = np.zeros((len(df_eval), len(TARGET_LABELS)))

for j, label in enumerate(TARGET_LABELS):
    eval_proba[:, j] = eval_proba_raw[:, class_to_col[label]]

pred_idx = np.argmax(eval_proba, axis=1)
y_pred = np.array(TARGET_LABELS)[pred_idx]

df_eval["predicted_label"] = y_pred
df_eval["predicted_confidence"] = np.max(eval_proba, axis=1)

for i, label in enumerate(TARGET_LABELS):
    df_eval[f"{label}_proba"] = eval_proba[:, i]

print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=TARGET_LABELS,
    zero_division=0
))

# sklearn top_k_accuracy_score requires labels to be ordered
ORDERED_LABELS = sorted(TARGET_LABELS)

# Re-align probability columns to ORDERED_LABELS
ordered_eval_proba = np.zeros((len(df_eval), len(ORDERED_LABELS)))

for j, label in enumerate(ORDERED_LABELS):
    original_col_idx = TARGET_LABELS.index(label)
    ordered_eval_proba[:, j] = eval_proba[:, original_col_idx]

print("\nTop-2 accuracy:", round(
    top_k_accuracy_score(
        y_true,
        ordered_eval_proba,
        k=2,
        labels=ORDERED_LABELS
    ),
    3
))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=TARGET_LABELS)
cm_df = pd.DataFrame(cm, index=TARGET_LABELS, columns=TARGET_LABELS)
print(cm_df)

Accuracy: 0.43333333333333335

Classification Report:
                   precision    recall  f1-score   support

      ideological       0.38      0.56      0.45        18
institutionalized       0.40      0.18      0.25        11
    interpersonal       0.38      0.40      0.39        15
     internalized       0.62      0.50      0.55        16

         accuracy                           0.43        60
        macro avg       0.44      0.41      0.41        60
     weighted avg       0.45      0.43      0.43        60


Top-2 accuracy: 0.833

Confusion Matrix:
                   ideological  institutionalized  interpersonal  internalized
ideological                 10                  1              5             2
institutionalized            6                  2              3             0
interpersonal                6                  0              6             3
internalized                 4                  2              2             8


In [142]:
# Eval Linear SVC

X_eval_svc = final_svd_svc.transform(X_eval_raw)

y_true = df_eval["primary_leaning"].values
y_pred_svc = final_svc.predict(X_eval_svc)

# LinearSVC does not have predict_proba()
# decision_function gives class scores instead
svc_scores_raw = final_svc.decision_function(X_eval_svc)

# Align score columns to TARGET_LABELS order
class_to_col = {label: i for i, label in enumerate(final_svc.classes_)}

svc_scores = np.zeros((len(df_eval), len(TARGET_LABELS)))

for j, label in enumerate(TARGET_LABELS):
    svc_scores[:, j] = svc_scores_raw[:, class_to_col[label]]

# sklearn top_k_accuracy_score requires labels to be ordered
ORDERED_LABELS = sorted(TARGET_LABELS)

ordered_svc_scores = np.zeros((len(df_eval), len(ORDERED_LABELS)))

for j, label in enumerate(ORDERED_LABELS):
    original_col_idx = TARGET_LABELS.index(label)
    ordered_svc_scores[:, j] = svc_scores[:, original_col_idx]

print("Accuracy:", accuracy_score(y_true, y_pred_svc))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred_svc,
    labels=TARGET_LABELS,
    zero_division=0
))

print("\nTop-2 accuracy:", round(
    top_k_accuracy_score(
        y_true,
        ordered_svc_scores,
        k=2,
        labels=ORDERED_LABELS
    ),
    3
))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred_svc, labels=TARGET_LABELS)
cm_df = pd.DataFrame(cm, index=TARGET_LABELS, columns=TARGET_LABELS)
print(cm_df)

Accuracy: 0.4166666666666667

Classification Report:
                   precision    recall  f1-score   support

      ideological       0.30      0.33      0.32        18
institutionalized       0.50      0.27      0.35        11
    interpersonal       0.40      0.53      0.46        15
     internalized       0.57      0.50      0.53        16

         accuracy                           0.42        60
        macro avg       0.44      0.41      0.41        60
     weighted avg       0.43      0.42      0.42        60


Top-2 accuracy: 0.867

Confusion Matrix:
                   ideological  institutionalized  interpersonal  internalized
ideological                  6                  1              8             3
institutionalized            5                  3              2             1
interpersonal                5                  0              8             2
internalized                 4                  2              2             8


In [143]:
# ------------------------------------------------------------
# Final small MLP trained on ALL generated sentences
# ------------------------------------------------------------

label_encoder = LabelEncoder()

X_full = X_full.astype("float32")
X_eval_raw = X_eval_raw.astype("float32")

y_train_full = df["Label"].values
y_train_full_encoded = label_encoder.fit_transform(y_train_full)


final_mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(256, 128, 64, 32),
        activation="relu",
        alpha=0.015,
        learning_rate_init=0.00025,
        max_iter=6000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=60,
        random_state=RANDOM_STATE
    ))
])


final_mlp.fit(X_full, y_train_full_encoded)

print("Final MLP trained on all generated sentences.")
print("Classes:", label_encoder.classes_)

Final MLP trained on all generated sentences.
Classes: ['ideological' 'institutionalized' 'internalized' 'interpersonal']


In [144]:
# ------------------------------------------------------------
# Evaluate MLP on real sentences
# ------------------------------------------------------------

y_true = df_eval["primary_leaning"].values
y_pred_mlp_encoded = final_mlp.predict(X_eval_raw)
y_pred_mlp = label_encoder.inverse_transform(y_pred_mlp_encoded)

print("Accuracy:", accuracy_score(y_true, y_pred_mlp))
print("Macro F1:", f1_score(y_true, y_pred_mlp, average="macro", zero_division=0))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred_mlp,
    labels=TARGET_LABELS,
    zero_division=0
))

print("\nConfusion Matrix:")
pd.DataFrame(
    confusion_matrix(y_true, y_pred_mlp, labels=TARGET_LABELS),
    index=TARGET_LABELS,
    columns=TARGET_LABELS
)

Accuracy: 0.43333333333333335
Macro F1: 0.4192546583850932

Classification Report:
                   precision    recall  f1-score   support

      ideological       0.32      0.50      0.39        18
institutionalized       0.67      0.18      0.29        11
    interpersonal       0.47      0.53      0.50        15
     internalized       0.58      0.44      0.50        16

         accuracy                           0.43        60
        macro avg       0.51      0.41      0.42        60
     weighted avg       0.49      0.43      0.43        60


Confusion Matrix:


,ideological,institutionalized,interpersonal,internalized
ideological,9,1,4,4
institutionalized,6,2,3,0
interpersonal,6,0,8,1
internalized,7,0,2,7


## Trying bert

In [145]:
LABELS = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]

MODEL_NAME = "bert-base-uncased"

In [146]:
df["sample_weight"] = 1.0

In [147]:
label2id = {
    "ideological": 0,
    "institutionalized": 1,
    "interpersonal": 2,
    "internalized": 3
}

id2label = {
    0: "ideological",
    1: "institutionalized",
    2: "interpersonal",
    3: "internalized"
}

df["label_id"] = df["Label"].map(label2id)
df_eval["label_id"] = df_eval["primary_leaning"].map(label2id)

In [148]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, dataframe, use_weights=True):
        self.sentences = dataframe["Sentence"].tolist()
        self.labels = dataframe["label_id"].values.astype("int64")

        if use_weights:
            self.weights = dataframe["sample_weight"].values.astype("float32")
        else:
            self.weights = np.ones(len(dataframe), dtype="float32")

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.sentences[idx],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "sample_weight": torch.tensor(self.weights[idx])
        }

        return item

train_ds = TextDataset(df, use_weights=True)
eval_ds = TextDataset(df_eval, use_weights=False)

In [149]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [150]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        weights = inputs.pop("sample_weight").float()

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
        loss = loss_fn(logits, labels)

        loss = (loss * weights).sum() / weights.sum()

        return (loss, outputs) if return_outputs else loss

In [151]:
args = TrainingArguments(
    output_dir="./bert_weighted",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds
)

trainer.train()

Step,Training Loss
20,1.382775
40,1.289013
60,1.161802
80,0.949321
100,0.813523
120,0.714308
140,0.627825
160,0.571223
180,0.485476
200,0.402214


TrainOutput(global_step=276, training_loss=0.7085765790248263, metrics={'train_runtime': 50.4039, 'train_samples_per_second': 43.647, 'train_steps_per_second': 5.476, 'total_flos': 144713679052800.0, 'train_loss': 0.7085765790248263, 'epoch': 4.0})

In [152]:
preds = trainer.predict(eval_ds)

logits = preds.predictions

pred_ids = logits.argmax(axis=1)

id2label = {
    0: "ideological",
    1: "institutionalized",
    2: "interpersonal",
    3: "internalized"
}

y_pred = [id2label[i] for i in pred_ids]

y_true = df_eval["primary_leaning"].values


print(classification_report(
    y_true,
    y_pred,
    target_names=LABELS,
    zero_division=0
))

                   precision    recall  f1-score   support

      ideological       0.37      0.56      0.44        18
institutionalized       1.00      0.27      0.43        11
    interpersonal       0.47      0.56      0.51        16
     internalized       0.45      0.33      0.38        15

         accuracy                           0.45        60
        macro avg       0.57      0.43      0.44        60
     weighted avg       0.53      0.45      0.45        60

